# Titanic Dataset ML Project
Dataset from Kaggle [Titanic Challenge](https://www.kaggle.com/competitions/titanic/data)

## Setup

### Import Packages

In [ ]:
from tqdm.auto import tqdm
import sklearn as sk
import pandas as pd
import numpy as np
import xgboost
import logging
import optuna
import re
import os

# For tqdm output to be the only cell output during hyperparam optimization.
optuna.logging.get_logger("optuna").setLevel(logging.WARNING)

## Dataset

### Import and Data Cleaning Function:

In [ ]:
def xy_dataset(filepath: str, both_xy: bool):
    '''
    Takes in dataset filepath 
    Returns X, and y, (shuffled and ready to be fitted)
    '''
    ## Importing data with pandas, dropping 
    DATA = pd.read_csv(filepath) # prev: ('data/train.csv')

    if both_xy:
        y = DATA['Survived'] # To be predicted
        # Remove columns that can (should) not be predictors
        X = DATA.drop(['Survived'], axis=1)
    else: X = DATA

    ## Clean dataset check, for debugging only
    def dataset_check(X_data):
        '''To check for the number of NA/empty values in any column'''
        check_empties = []
        for column in X_data.columns:
            tmp_list = [column, int(np.sum(X_data[column].isna()))]
            #print(column)
            #print(np.sum(X_data[column].isna()))
            check_empties.append(tmp_list)
        print(check_empties)

    # Uncomment below to check which columns have missing/NA values
    #dataset_check(X)

    ## Impute missing age values as median of Name title (Mr. Mrs. Ms. ...), Pclass (1, 2, 3), and Sex (Male, Female)
    def impute_age(X_data):
        '''
        Imputing missing age as median of: Name's title/honorific,
        Pclass (social standing), and Sex.
        '''
        ## Find all unique titles/honorifics:
        # Assume:   Title starts after ", " and ends after ".", 
        #           also assume no missing names in dataset
        # Strategy: Strip all names down using regex and lstrip/rstrip
        Titles = []
        for name in X_data['Name']: # For all names in the dataset
            match = re.search(", .*?\\. ",name).group() # The part of the string that matches our conditions to start at ', ' and end at '. '
            Titles.append(match.lstrip(', ').rstrip(' ')) # Append the honorific stripped of prefix/suffix
        UTitles = set(Titles) # All unique titles

        # To check, uncomment below
        #print(len(UTitles)," unique titles / honorifics")
        #print(UTitles)

        ## Imputing age values by sex, social status, then by title/honorific
        
        # For cases where these subdivisions do exist
        unique_titles = X_data['Name'].str.extract(f"({'|'.join(UTitles)})", expand=False) # Searching for any titles in the unique list
        X_data['Age'] = X_data['Age'].fillna(X_data.groupby(['Sex', 'Pclass', unique_titles])['Age'].transform('median')) 

        # If no other examples within sex/social status/title exist, impute those rows' age with global median
        global_median = X_data['Age'].median()
        X_data['Age'] = X_data['Age'].fillna(global_median)

        return X_data

    ## Drop irrelevant columns
    def drop_cols(X_data):
        '''
        Drop irrelevant columns in input data matrix. (Placed in a function to alter as necessary)
        Placed after Age imputation as title / honorific extracted from inside Name.
        '''
        X_data = X_data.drop(columns=['Name','Cabin','PassengerId','Ticket']) # Passenger ID needs to be saved later for expected Kaggle submission
        return X_data # Could also try creating dummies from the titles/honorifics from Names but this may cause model memorization 

    # Encode categrocial columns into dummy columns so numeric-centric models will take the data 
    def encoding_multi_categories(X_data):
        '''
        Imputing missing Embarked entries with the mode.
        Encoding categorical data as numerical values and dummy columns so numeric models are able to function appropriately.
        '''
        # Encode sex with pd dummies

        # Imputing embarked categorical with most often occuring value, no missing values for sex (excluded)
        X_data['Embarked'] = X_data['Embarked'].fillna(X_data['Embarked'].mode()[0]) 

        # Convert categoricals to OHE columns
        X_data_encoded = pd.get_dummies(X_data, columns=['Embarked','Sex'], drop_first=True) # Try with changing drop_true to False, likely no change

        # Convert T/F to 1/0
        X_data_encoded = X_data_encoded.astype(float)

        return X_data_encoded


    ## Applying missing Age values imputation
    X = impute_age(X)

    ## Dropping irrelevant columns
    X = drop_cols(X)

    ## Encoding categorical columns to dummy columns with numerical values for True/False
    X = encoding_multi_categories(X)

    if both_xy:
        ## Shuffling training examples
        X, y = sk.utils.shuffle(X,y,random_state=None) 
        return X, y
    else: return X  # No need to shuffle test set, doesn't affect anything and maintaining 
                    # order will be used downstream to recombine dataframe for submission file.

### Applying above function for training data

In [3]:
X_train, y_train = xy_dataset('data/train.csv', both_xy=True)

### Applying function for test data 

In [ ]:
X_test = xy_dataset('data/test.csv', both_xy=False)

## Modelling Data 

### Model Dictionary Setup

In [6]:
# Models used (Estimators dict / list)
# Use: Random Forest, XGBoost, LightGBM (Not the last one)
estimators = {
    #[name, model]
    'random_forest':   sk.ensemble.RandomForestClassifier,
    'xgboost':         xgboost.XGBClassifier 
    }

#estimators.items()

### Unoptimized Training

In [7]:
# model.fit(X_train,y_train)
models = {}

for model_name, model in estimators.items():
    #print(model_name)
    trained = model().fit(X_train,y_train)
    models[model_name] = trained
    trained = None
    #print(model)

### Unoptimized Results

In [ ]:
# Scores displayed for; accuracy, AUC, classification report table summary
# model.predict(x_test)
# sklearn.metrics.score(y_train,y_test)

print("Where 1 represents an individual passenger's survival "
        "and 0 represents a single passenger not surviving:\n")

CLASS_REPORT_DICT = {}

for model_name in models:
    y_pred = models[model_name].predict(X_train)

    ## Uncomment to get bulk results using the standard classification_report  
    #class_rep = sk.metrics.classification_report(y_train,y_pred)
    #CLASS_REPORT_DICT[model_name] = sk.metrics.classification_report(y_train,y_pred,output_dict=True)
    #print(f"Classification Report for {model_name.replace("_"," ").capitalize()} model:\n{class_rep}")
    
    # Viewing accuracy as the sole displayed model metric, chose this as accuracy-only 
    # follows the Kaggle challenge's scoring assumption though other metrics are available above and below
    accuracy = sk.metrics.accuracy_score(y_train, y_pred)
    print("Accuracy of",model_name,"model: ",accuracy)

    # Area under the curve also a possible metric to output
    #AUC = sk.metrics.auc(y_train,y_pred)
    #print("AUC of",model_name,"model: ",AUC)

Where 0 represents an individual passenger's survival and 1 represents a single passenger not surviving:

Accuracy of random_forest model:  0.9820426487093153
Accuracy of xgboost model:  0.9719416386083053


## Hyperparameter Optimization

### Optuna Objective Function
Takes in an optuna trial object, sets up and trains ML model, returns evaluation metric(s)

In [ ]:
def objective(trial):
    X, y = X_train, y_train
    
    model_name = trial.suggest_categorical("Classifier", ["random_forest","xgboost"])

    if model_name == 'random_forest':
        params = {
            'n_estimators': trial.suggest_int(name="Number of Estimators",low=100,high=1000,step=100),
            'max_depth': trial.suggest_int(name="RF Maximum Depth",low=1,high=301,step=10),
            'max_features': trial.suggest_categorical("RF Max Features",["sqrt","log2"])
            }
    elif model_name == 'xgboost':
        params = {
            'learning_rate': trial.suggest_float(name="Learning Rate",low=0.1,high=0.9),
            'gamma': trial.suggest_float(name="Minimum Split Loss",low=0,high=0.8),
            'max_depth': trial.suggest_int(name="XGB Maximum Depth",low=2,high=18)
            }
    else: return AssertionError("Model used not in estimators list")

    clf = estimators[model_name](**params).fit(X,y)

    y_pred = clf.predict(X)

    # Accuracy across 'cv' folds, but just fitting on raw training data
    #score = sk.model_selection.cross_val_score(clf,X,y,cv=5).mean() # Commenting out as may cause overfit, may also prevent overfit...

    # Just sticking with accuracy_score to conform to Kaggle specifications
    score = sk.metrics.accuracy_score(y,y_pred) # To more closely analyze wrt unoptimized results
    
    return score


### Hyperparameter Optimization Study

In [10]:
N_TRIALS = 1000

# Using TQDM to visualize training progress
with tqdm(total=N_TRIALS,desc="Optimizing Models",unit="trial") as pbar:

    def tqdm_callback(study,trial):
        if study.best_trials:
            pbar.set_postfix({"Best Score": f"{study.best_value:.4f}"})
        pbar.update(1)

    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(n_startup_trials=10))
    study.optimize(objective,n_trials=N_TRIALS,callbacks=[tqdm_callback])

Optimizing Models:   0%|          | 0/1000 [00:00<?, ?trial/s]

### Results of Best Optimized Model

In [ ]:
print(f"Best optimized model performance: {study.best_value}\n"
      f"Best optimized model hyperparam configuration: {study.best_params}")

# Looks like unoptimized models are just as good, error may be attributable to a specific 
# example in the training set, causing 0.9820... to be the highest achievable accuracy

Best optimized model performance: 0.9820426487093153
Best optimized model hyperparam configuration: {'Classifier': 'random_forest', 'Number of Estimators': 1000, 'RF Maximum Depth': 151, 'RF Max Features': 'log2'}


In [13]:
study.best_params['Classifier']

'random_forest'

#### Training a Final Model Using the Best Hyperparameters from the Study 

In [28]:
if study.best_params['Classifier'] == 'random_forest':
    params = {
        'n_estimators': study.best_params["Number of Estimators"],
        'max_depth': study.best_params["RF Maximum Depth"],
        'max_features': study.best_params["RF Max Features"]
    }
    best_model = estimators['random_forest'](**params).fit(X_train,y_train)
    y_train_pred = best_model.predict(X_train)
    acc_score = sk.metrics.accuracy_score(y_train, y_train_pred)
elif study.best_params['Classifier'] == 'xgboost':
    params = {
        'learning_rate': study.best_params["Learning Rate"],
        'gamma': study.best_params["Minimum Split Loss"],
        'max_depth': study.best_params["XGB Maximum Depth"]
    }
    best_model = estimators['xgboost'](**params).fit(X_train,y_train)
    y_train_pred = best_model.predict(X_train)
    acc_score = sk.metrics.accuracy_score(y_train, y_train_pred)
else: raise ValueError("Best classifier's name not recognized")

print(f"Successfully trained {study.best_params['Classifier']} model using the following hyperparameters:\n{params}\n\
    With a training accuracy score of {acc_score:.4f}")

Successfully trained random_forest model using the following hyperparameters:
{'n_estimators': 1000, 'max_depth': 151, 'max_features': 'log2'}
    With a training accuracy score of 0.9820


## Test Set Prediction & Submission Preparation
Kaggle challenge does not provide ground truth results for test set data, final prediction submissions are expected in following format:
|PassengerId|Survived|
|:---:|:---:|
|int|bool _(numerical -> 1/0)_|

 Prepared submission file will be written to _output/submission.csv_

### Predicting Test Set with Best Optimized Model

In [38]:
# Predicting using most recently trained best_model
y_test_prediction = best_model.predict(X_test)

## Survival rate calculation:

# As 1 represents a survival, the sum of all 1's within the test set 
# predictions will be the number of survivals for the entire test set
num_survived = y_test_prediction.sum() 

survival_rate = num_survived/len(y_test_prediction)

print(f"The survival rate of the test set is ~{100*survival_rate:.2g}% of passengers survived")

# (Additional analysis)
# The difference between the number of passengers in the test set and 
# the number of survivals is the number of those who did not survive
num_not_survived = len(y_test_prediction)-num_survived 

survival_ratio = num_survived/num_not_survived

print(f"The ratio of survival for the test set is 1 passenger survived to every {1/survival_ratio:.2g} passengers that did not survive")

The survival rate of the test set is ~37% of passengers survived
The ratio of survival for the test set is 1 passenger survived to every 1.7 passengers that did not survive


### Expected Formatting for Kaggle Submission

In [57]:
# Recombining PassengerId to non-shuffled y_test_prediction output from model

# Re-importing for just the PassengerId since the data import & cleaning functon drops 
# the PassengerId column but does not shuffle for the test set 
test_PIds = pd.read_csv('data/test.csv')['PassengerId'].to_frame()
#print(test_PIds)

y_test_pred_df = pd.DataFrame(y_test_prediction,columns=["Survived"]) # Converting array to dataframe to merge
#print(y_test_pred_df)

# Merge with DF.join, inner works on the index if both DF's are 1D / only 1 column 
test_set_output = test_PIds.join(other=y_test_pred_df,how='inner') 
print(test_set_output)

     PassengerId  Survived
0            892         0
1            893         0
2            894         0
3            895         1
4            896         0
..           ...       ...
413         1305         0
414         1306         1
415         1307         0
416         1308         0
417         1309         1

[418 rows x 2 columns]


### Writeout to .csv

In [ ]:
if not os.path.exists('output'): os.makedirs('output') # Making output folder if it doesn't already exist 

test_set_output.to_csv('output/submission',index=False) # index=False ensures we dont explicitly include the row numbers into .csv